# CÓDIGO: Modelamiento por Random Forest


## ── Librerías

In [14]:
import pandas as pd
import numpy as np
import re
import nltk
import spacy
import warnings
warnings.filterwarnings("ignore")

from nltk.corpus import stopwords
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import sklearn.preprocessing
from sklearn.preprocessing import LabelEncoder

In [2]:
# Descargar recursos de NLTK
nltk.download("stopwords")
nltk.download("punkt")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dacma\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dacma\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
# Cargar modelo de spaCy en español
def cargar_modelo_spacy_es():
    for model_name in ("es_core_news_md", "es_core_news_sm"):
        try:
            print(f"Cargando modelo spaCy: {model_name}")
            return spacy.load(model_name)
        except OSError:
            continue

    print("No se encontro un modelo de spaCy en espanol instalado. Se usara un pipeline basico.")
    return spacy.blank("es")


nlp = cargar_modelo_spacy_es()

print("✅ Librerías cargadas correctamente.")

Cargando modelo spaCy: es_core_news_md
✅ Librerías cargadas correctamente.


## ── 2. CARGA DE DATOS

In [4]:

df = pd.read_excel("data/Data_arreglada.xlsx", sheet_name="Sheet1")

In [5]:
# Renombrar columnas para facilitar el trabajo
df.columns = ["Categoria", "Resuelta", "Consulta"]

print(f"📦 Shape original: {df.shape}")
print(df.head(5))

📦 Shape original: (36468, 3)
                                   Categoria Resuelta  \
0                              Carnetización       No   
1                       Calendario académico       No   
2                              Carnetización       Sí   
3  ​ Gestión Económica - Información general       Sí   
4                              Carnetización       No   

                                            Consulta  
0  Ayuda no se como inscribir asignaturas y nadie...  
1           Donde puedo ver el calendario académico?  
2                                                NaN  
3                                                NaN  
4                              Quiero mi carné nuevo  


## ── 3. LIMPIEZA INICIAL 

In [6]:
# Eliminar filas donde Consulta o Categoria sean nulas
df = df.dropna(subset=["Consulta", "Categoria"])

In [7]:
# Eliminar filas que no fueron resueltas (Resuelta = 'Sí')
df = df[df["Resuelta"].str.strip().str.lower() != "sí"]
print(f"\n📦 Shape después de eliminar resueltas: {df.shape}")


📦 Shape después de eliminar resueltas: (35168, 3)


In [8]:
# Eliminar filas con consulta vacía o solo espacios
df = df[df["Consulta"].str.strip() != ""]
df = df.reset_index(drop=True)

In [9]:
print(f" Shape final del dataset: {df.shape}")
print(df.head(15))

 Shape final del dataset: (35168, 3)
                                            Categoria Resuelta  \
0                                       Carnetización       No   
1                                Calendario académico       No   
2                                       Carnetización       No   
3                                Calendario académico       No   
4                                Calendario académico       No   
5           ​ Gestión Económica - Información general       No   
6                                Calendario académico       No   
7           ​ Gestión Económica - Información general       No   
8                                Calendario académico       No   
9                                       Carnetización       No   
10                               Calendario académico       No   
11                                      Carnetización       No   
12          ​ Gestión Económica - Información general       No   
13  ​ Gestión Económica - Inconsistenci

## ── 4. MAPEO DE CATEGORÍAS VÁLIDAS

In [10]:
CATEGORIAS_VALIDAS = [
    "Carnetización",
    "Actualización de datos personales",
    "Calendario académico",
    "Certificados",
    "Gestión Académica",
    "Gestión Económica",
    "Reubicación socioeconómica en Pregrado",
    "Aplazamiento de matrícula inicial",
    "Política de gratuidad (matrícula cero) Pregrado",
    "Información general sobre servicios estudiantiles",
]

def mapear_categoria(categoria: str) -> str:
    """
    Mapea cada categoría original del dataset a una de las CATEGORIAS_VALIDAS.
    Usa coincidencia parcial por palabras clave.
    """
    categoria = str(categoria).strip()

    # Mapeos directos y por palabras clave
    mapeo = {
        "Carnetización":                                    ["carnetización", "carnet", "carné"],
        "Actualización de datos personales":                ["actualización de datos", "datos personales", "actualización"],
        "Calendario académico":                             ["calendario académico", "calendario"],
        "Certificados":                                     ["certificado"],
        "Gestión Académica":                                ["gestión académica", "inscripción", "adiciones",
                                                             "cancelaciones", "asignaturas", "sobrecupo",
                                                             "historia académica", "bloqueo", "grado",
                                                             "homologación", "traslado", "aplazamiento",
                                                             "reingreso", "notas", "prueba", "inglés",
                                                             "doble titulación", "posgrado"],
        "Gestión Económica":                                ["gestión económica", "recibo", "pago",
                                                             "fraccionamiento", "unificación", "devolución",
                                                             "financiación", "matrícula", "pbm",
                                                             "descuento", "electoral", "generación e",
                                                             "icetex", "ser pilo", "exención",
                                                             "reexpedición", "cobro", "deuda"],
        "Reubicación socioeconómica en Pregrado":           ["reubicación socioeconómica", "reubicación",
                                                             "socioeconómica", "socioeconómico"],
        "Aplazamiento de matrícula inicial":                ["aplazamiento de matrícula", "aplazamiento inicial",
                                                             "aplazamiento"],
        "Política de gratuidad (matrícula cero) Pregrado": ["matrícula cero", "gratuidad", "matrícula 0",
                                                             "política de gratuidad"],
        "Información general sobre servicios estudiantiles":["información general", "información financiera",
                                                              "servicios estudiantiles", "bienestar",
                                                              "alimentaria", "correo institucional",
                                                              "sia", "bicirún", "sibu"],
    }

    categoria_lower = categoria.lower()

    for cat_valida, palabras_clave in mapeo.items():
        for palabra in palabras_clave:
            if palabra in categoria_lower:
                return cat_valida

    # Si no hay coincidencia, intentar por la consulta (fallback)
    return "Información general sobre servicios estudiantiles"


# Aplicar el mapeo
df["Categoria_Mapeada"] = df["Categoria"].apply(mapear_categoria)

print("\n📊 Distribución de categorías mapeadas:")
print(df["Categoria_Mapeada"].value_counts())


📊 Distribución de categorías mapeadas:
Categoria_Mapeada
Gestión Económica                                    15751
Gestión Académica                                    10520
Información general sobre servicios estudiantiles     4714
Certificados                                          2656
Actualización de datos personales                      768
Carnetización                                          458
Calendario académico                                   232
Reubicación socioeconómica en Pregrado                  49
Política de gratuidad (matrícula cero) Pregrado         20
Name: count, dtype: int64


## ── 5. PREPROCESAMIENTO DE TEXTO

In [11]:
STOPWORDS_ES = set(stopwords.words("spanish"))

# Stopwords adicionales específicas del dominio universitario
STOPWORDS_EXTRA = {
    "universidad", "nacional", "colombia", "unal", "sede", "bogotá",
    "favor", "gracias", "buenas", "buenos", "días", "tardes", "noches",
    "cordial", "saludo", "atentamente", "amablemente", "presente",
    "correo", "motivo", "solicito", "solicitud", "quisiera", "quiero",
    "necesito", "requiero", "agradezco", "agradecería", "muchas",
    "manera", "forma", "caso", "parte", "vez", "día", "semestre",
    "periodo", "académico", "estudiante", "programa", "curricular",
    "sia", "dninfoa", "portal", "plataforma", "sistema",
}

STOPWORDS_COMPLETO = STOPWORDS_ES.union(STOPWORDS_EXTRA)


def limpiar_texto(texto: str) -> str:
    """Limpieza básica: minúsculas, eliminar caracteres especiales y números."""
    texto = str(texto).lower()
    # Eliminar URLs
    texto = re.sub(r"http\S+|www\S+", " ", texto)
    # Eliminar correos electrónicos
    texto = re.sub(r"\S+@\S+", " ", texto)
    # Eliminar números y caracteres especiales, conservar letras y espacios
    texto = re.sub(r"[^a-záéíóúüñ\s]", " ", texto)
    # Eliminar espacios múltiples
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def eliminar_stopwords(texto: str) -> str:
    """Elimina stopwords del texto."""
    tokens = texto.split()
    tokens_filtrados = [t for t in tokens if t not in STOPWORDS_COMPLETO and len(t) > 2]
    return " ".join(tokens_filtrados)


def lematizar(texto: str) -> str:
    """Lematiza el texto usando spaCy."""
    doc = nlp(texto)
    lemas = [
        token.lemma_ if token.lemma_ else token.text
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and len(token.lemma_ if token.lemma_ else token.text) > 2
    ]
    return " ".join(lemas)


def preprocesar(texto: str) -> str:
    """Pipeline completo: limpieza → stopwords → lematización."""
    texto = limpiar_texto(texto)
    texto = eliminar_stopwords(texto)
    texto = lematizar(texto)
    return texto


print("\n⚙️  Aplicando preprocesamiento (puede tardar unos minutos)...")
df["Consulta_Procesada"] = df["Consulta"].apply(preprocesar)

print("✅ Preprocesamiento completado.")
print("\nEjemplo de transformación:")
print(f"  ORIGINAL : {df['Consulta'].iloc[0][:100]}...")
print(f"  PROCESADO: {df['Consulta_Procesada'].iloc[0][:100]}...")


⚙️  Aplicando preprocesamiento (puede tardar unos minutos)...
✅ Preprocesamiento completado.

Ejemplo de transformación:
  ORIGINAL : Ayuda no se como inscribir asignaturas y nadie me explica....
  PROCESADO: ayuda inscribir asignatura explicar...


## ── 6. PREPARACIÓN DE FEATURES Y ETIQUETAS

In [12]:
# Eliminar filas con texto procesado vacío
df = df[df["Consulta_Procesada"].str.strip() != ""]
df = df.reset_index(drop=True)

X = df["Consulta_Procesada"]
y = df["Categoria_Mapeada"]

# Codificar etiquetas
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"\n📦 Total de muestras para entrenamiento: {len(X)}")
print(f"🏷️  Clases: {list(le.classes_)}")


📦 Total de muestras para entrenamiento: 35150
🏷️  Clases: ['Actualización de datos personales', 'Calendario académico', 'Carnetización', 'Certificados', 'Gestión Académica', 'Gestión Económica', 'Información general sobre servicios estudiantiles', 'Política de gratuidad (matrícula cero) Pregrado', 'Reubicación socioeconómica en Pregrado']


## ── 7. VECTORIZACIÓN TF-IDF

In [15]:
tfidf = TfidfVectorizer(
    max_features=3000,       # Máximo 5000 términos
    ngram_range=(1, 2),      # Unigramas y bigramas
    min_df=2,                # Mínimo 2 documentos
    sublinear_tf=True,       # Escala logarítmica
)

X_tfidf = tfidf.fit_transform(X)
#matriz densa para mostrar dimensiones
X_dense = X_tfidf.todense()
print(f"\n🔢 Matriz TF-IDF: {X_dense.shape}")


🔢 Matriz TF-IDF: (35150, 3000)


## ── 8. DIVISIÓN TRAIN / TEST

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X_dense, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded,
)

print(f"\n📊 Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")


📊 Train: 28120 muestras | Test: 7030 muestras


## ── 9. ENTRENAMIENTO DEL MODELO 

In [18]:
modelo_rf = RandomForestClassifier(
    n_estimators=200,        # Número de árboles
    max_depth=30,            # Profundidad máxima por árbol
    min_samples_split=4,     # Mínimo de muestras para dividir un nodo
    min_samples_leaf=2,      # Mínimo de muestras en hoja
    max_features="sqrt",     # Features por árbol: raíz cuadrada del total
    class_weight="balanced", # Maneja desbalance de clases
    random_state=42,
    n_jobs=-1,               # Usa todos los núcleos disponibles
    verbose=1,
)

print("\n Entrenando Random Forest...")
X_train = np.asarray(X_train)
X_test = np.asarray(X_test)

modelo_rf.fit(X_train, y_train)
modelo = modelo_rf
print("✅ Modelo Random Forest entrenado correctamente.")


 Entrenando Random Forest...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    3.0s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:   16.7s


✅ Modelo Random Forest entrenado correctamente.


[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:   18.5s finished


## ── 10. EVALUACIÓN

In [20]:
y_pred = modelo_rf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\n🎯 Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Validación cruzada (5 folds) para una estimación más robusta
print("\n🔄 Calculando validación cruzada (5 folds)...")
# Convertir np.matrix -> np.ndarray para evitar el error en sklearn
X_cv = np.asarray(X_dense)

cv_scores = cross_val_score(
    modelo_rf, X_cv, y_encoded,
    cv=5, scoring="accuracy", n_jobs=-1,
    error_score="raise"
)
print(f"📊 CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"   Scores por fold: {[round(s, 4) for s in cv_scores]}")

print("\n📋 Reporte de Clasificación:")
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    zero_division=0,
))

print("\n🔲 Matriz de Confusión:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
print(cm_df)

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.1s finished



🎯 Accuracy: 0.6942 (69.42%)

🔄 Calculando validación cruzada (5 folds)...
📊 CV Accuracy: 0.6972 ± 0.0152
   Scores por fold: [np.float64(0.6782), np.float64(0.6947), np.float64(0.6839), np.float64(0.7154), np.float64(0.7139)]

📋 Reporte de Clasificación:
                                                   precision    recall  f1-score   support

                Actualización de datos personales       0.46      0.65      0.54       153
                             Calendario académico       0.17      0.39      0.24        46
                                    Carnetización       0.61      0.90      0.73        92
                                     Certificados       0.70      0.93      0.80       531
                                Gestión Académica       0.78      0.50      0.61      2102
                                Gestión Económica       0.82      0.83      0.82      3150
Información general sobre servicios estudiantiles       0.40      0.53      0.45       942
  Política de g

## ── 11. IMPORTANCIA DE FEATURES

In [21]:
# Característica exclusiva de Random Forest vs Regresión Logística
importancias = modelo_rf.feature_importances_
nombres_features = tfidf.get_feature_names_out()

df_importancias = pd.DataFrame({
    "Feature": nombres_features,
    "Importancia": importancias
}).sort_values("Importancia", ascending=False)

print("\n🔑 Top 20 palabras más importantes para la clasificación:")
print(df_importancias.head(20).to_string(index=False))


🔑 Top 20 palabras más importantes para la clasificación:
                   Feature  Importancia
               certificado     0.049983
            socioeconómico     0.029613
                    carnet     0.028976
     certificado electoral     0.027709
                 electoral     0.026466
               reubicación     0.025005
                    recibo     0.022461
                 gratuidad     0.021886
                      pago     0.018213
             carnetización     0.016404
                calendario     0.016171
reubicación socioeconómico     0.014411
             actualización     0.013822
                     carné     0.012159
                  político     0.011307
        político gratuidad     0.011161
                      dato     0.010897
                     fecha     0.010407
                      cita     0.010153
                actualizar     0.010089


## 12. FUNCIÓN DE PREDICCIÓN

In [23]:
def predecir_categoria(texto_nuevo: str) -> dict:
    """
    Predice la categoría de una nueva consulta con Random Forest.
    Retorna la categoría predicha, probabilidades y confianza.
    """
    texto_proc  = preprocesar(texto_nuevo)
    texto_vec = np.asarray(tfidf.transform([texto_proc]).todense())  # Dense para RF (evita error con toarray en spmatrix)
    pred_encoded    = modelo_rf.predict(texto_vec)[0]
    pred_categoria  = le.inverse_transform([pred_encoded])[0]
    probabilidades  = modelo_rf.predict_proba(texto_vec)[0]

    probs_dict = {
        clase: round(float(prob), 4)
        for clase, prob in zip(le.classes_, probabilidades)
    }
    probs_ordenadas = dict(
        sorted(probs_dict.items(), key=lambda x: x[1], reverse=True)
    )

    confianza = max(probabilidades)

    return {
        "categoria_predicha": pred_categoria,
        "confianza": round(float(confianza), 4),
        "probabilidades": probs_ordenadas,
    }

## 13. PRUEBA CON EJEMPLOS

In [24]:
ejemplos = [
    "No me aparece el recibo de pago en el SIA y necesito pagarlo urgente",
    "Quisiera saber cómo inscribir materias para este semestre",
    "Necesito tramitar mi carné universitario",
    "Soy estrato 2 y quiero saber si aplico para matrícula cero",
    "Solicito certificado de notas para trámite externo",
    "Quiero aplazar mi matrícula por motivos económicos",
    "Mi PBM quedó muy alto y no tengo recursos para pagar",
    "No me asignaron cita de inscripción de asignaturas",
]

print("\n🧪 PRUEBAS DE PREDICCIÓN:")
print("=" * 65)
for ejemplo in ejemplos:
    resultado = predecir_categoria(ejemplo)
    print(f"\n📝 Consulta   : {ejemplo}")
    print(f"🏷️  Predicción : {resultado['categoria_predicha']}")
    print(f"🔒 Confianza  : {resultado['confianza']*100:.1f}%")
    top3 = list(resultado["probabilidades"].items())[:3]
    print(f"📊 Top-3 probs: {top3}")
print("=" * 65)


🧪 PRUEBAS DE PREDICCIÓN:


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished



📝 Consulta   : No me aparece el recibo de pago en el SIA y necesito pagarlo urgente
🏷️  Predicción : Gestión Económica
🔒 Confianza  : 64.7%
📊 Top-3 probs: [('Gestión Económica', 0.6468), ('Gestión Académica', 0.1193), ('Información general sobre servicios estudiantiles', 0.0825)]

📝 Consulta   : Quisiera saber cómo inscribir materias para este semestre
🏷️  Predicción : Gestión Académica
🔒 Confianza  : 35.8%
📊 Top-3 probs: [('Gestión Académica', 0.3582), ('Información general sobre servicios estudiantiles', 0.229), ('Calendario académico', 0.1236)]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Do


📝 Consulta   : Necesito tramitar mi carné universitario
🏷️  Predicción : Carnetización
🔒 Confianza  : 43.0%
📊 Top-3 probs: [('Carnetización', 0.4299), ('Información general sobre servicios estudiantiles', 0.1375), ('Gestión Académica', 0.104)]

📝 Consulta   : Soy estrato 2 y quiero saber si aplico para matrícula cero
🏷️  Predicción : Gestión Académica
🔒 Confianza  : 19.9%
📊 Top-3 probs: [('Gestión Académica', 0.1987), ('Información general sobre servicios estudiantiles', 0.1897), ('Gestión Económica', 0.1844)]


[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s



📝 Consulta   : Solicito certificado de notas para trámite externo
🏷️  Predicción : Certificados
🔒 Confianza  : 56.0%
📊 Top-3 probs: [('Certificados', 0.56), ('Información general sobre servicios estudiantiles', 0.1062), ('Gestión Académica', 0.0783)]

📝 Consulta   : Quiero aplazar mi matrícula por motivos económicos
🏷️  Predicción : Información general sobre servicios estudiantiles
🔒 Confianza  : 20.6%
📊 Top-3 probs: [('Información general sobre servicios estudiantiles', 0.2064), ('Gestión Académica', 0.1508), ('Gestión Económica', 0.1418)]


[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished



📝 Consulta   : Mi PBM quedó muy alto y no tengo recursos para pagar
🏷️  Predicción : Información general sobre servicios estudiantiles
🔒 Confianza  : 20.3%
📊 Top-3 probs: [('Información general sobre servicios estudiantiles', 0.2027), ('Gestión Económica', 0.1884), ('Gestión Académica', 0.172)]

📝 Consulta   : No me asignaron cita de inscripción de asignaturas
🏷️  Predicción : Gestión Académica
🔒 Confianza  : 37.4%
📊 Top-3 probs: [('Gestión Académica', 0.3743), ('Calendario académico', 0.2077), ('Información general sobre servicios estudiantiles', 0.1459)]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished


## 3. GUARDAR RESULTADOS

In [26]:
df_resultado = df[[
    "Consulta", "Categoria", "Categoria_Mapeada", "Consulta_Procesada"
]].copy()

# Evitar np.matrix: usar np.ndarray para la predicción
df_resultado["Prediccion"] = le.inverse_transform(
    modelo_rf.predict(np.asarray(X_dense))
)
df_resultado["Correcto"] = (
    df_resultado["Categoria_Mapeada"] == df_resultado["Prediccion"]
)

df_resultado.to_excel("resultados_random_forest.xlsx", index=False)
df_importancias.head(50).to_excel("importancia_features_rf.xlsx", index=False)

print("\n💾 Archivos guardados:")
print("   → resultados_random_forest.xlsx")
print("   → importancia_features_rf.xlsx")

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.7s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.8s finished



💾 Archivos guardados:
   → resultados_random_forest.xlsx
   → importancia_features_rf.xlsx
